# Mathematical Tools in RustQuant

RustQuant includes a comprehensive math module with:

- **Numerical integration** (Tanh-Sinh quadrature)
- **Interpolation** (Linear, Exponential)
- **Optimization** (Gradient Descent, Newton-Raphson)
- **Root-finding** (Bisection, Brent's method)
- **Sequences** (linspace, seq, cumsum, rep)
- **Curves** (yield curves with interpolation)

## Setup

In [ ]:
:dep RustQuant = { path = "../crates/RustQuant" }
:dep time = { version = "0.3", features = ["macros"] }

## 1. Numerical Integration

RustQuant uses the Tanh-Sinh (double exponential) quadrature method,
which is highly accurate for smooth functions.

In [ ]:
use std::f64::consts::PI;
use RustQuant::math::*;

// Integrate the standard normal PDF from -5 to 5
fn normal_pdf(x: f64) -> f64 {
    (2.0 * PI).sqrt().recip() * (-0.5 * x.powi(2)).exp()
}

let result = integrate(normal_pdf, -5.0, 5.0);
println!("Integral of N(0,1) PDF from -5 to 5 = {:.10} (expected ~1.0)", result);

// Integrate sin(x) from 0 to pi
let result = integrate(f64::sin, 0.0, PI);
println!("Integral of sin(x) from 0 to pi    = {:.10} (expected 2.0)", result);

// Integrate x^2 from 0 to 1
let result = integrate(|x| x * x, 0.0, 1.0);
println!("Integral of x^2 from 0 to 1         = {:.10} (expected 1/3)", result);

## 2. Gradient Descent Optimization

Minimize functions using gradient descent with automatic differentiation.
RustQuant's autodiff engine computes exact gradients.

In [ ]:
use RustQuant::autodiff::*;
use RustQuant::math::optimization::gradient_descent::*;

// Himmelblau's function: f(x, y) = (x^2 + y - 11)^2 + (x + y^2 - 7)^2
// Has four minima, all with f = 0:
//   (3.0, 2.0), (-2.805, 3.131), (-3.779, -3.283), (3.584, -1.848)
fn himmelblau<'v>(vars: &[Variable<'v>]) -> Variable<'v> {
    let x = vars[0];
    let y = vars[1];
    (x.powf(2.0) + y - 11.0).powf(2.0) + (x + y.powf(2.0) - 7.0).powf(2.0)
}

let gd = GradientDescent::new(0.005, 200, None);

// From (5, 5) -> converges to (3, 2)
let result = gd.optimize(himmelblau, &[5.0, 5.0], false);
println!("Starting from (5, 5):");
println!("  Minimum at: ({:.4}, {:.4})", result.minimizer[0], result.minimizer[1]);

// From (-4, -4) -> converges to (-3.779, -3.283)
let result = gd.optimize(himmelblau, &[-4.0, -4.0], false);
println!("Starting from (-4, -4):");
println!("  Minimum at: ({:.4}, {:.4})", result.minimizer[0], result.minimizer[1]);

## 3. Yield Curve Construction

Build and interpolate yield curves from market data.

In [ ]:
use time::macros::date;
use time::Duration;
use RustQuant::data::*;

// US Treasury rates as of a sample date
let today = date!(2024 - 08 - 01);

let dates = vec![
    today + Duration::days(30),     // 1M
    today + Duration::days(90),     // 3M
    today + Duration::days(180),    // 6M
    today + Duration::days(365),    // 1Y
    today + Duration::days(365 * 2),// 2Y
    today + Duration::days(365 * 5),// 5Y
    today + Duration::days(365 * 10),// 10Y
    today + Duration::days(365 * 30),// 30Y
];

let rates = vec![5.55, 5.37, 5.08, 4.62, 4.16, 3.84, 3.99, 4.27];

let curve = Curve::new(dates, rates, CurveType::Spot, InterpolationMethod::Linear).unwrap();

// Interpolate rates at arbitrary dates
println!("Yield Curve Interpolation:");
for months in [2, 4, 9, 18, 36, 84] {
    let target_date = today + Duration::days(months * 30);
    if let Some(rate) = curve.get_rate(target_date) {
        println!("  {}M: {:.2}%", months, rate);
    }
}

## 4. Sequences and Utilities

RustQuant provides useful numerical sequence utilities.

In [ ]:
use RustQuant::math::*;

// linspace: generate evenly spaced values
let xs = f64::linspace(0.0, 1.0, 11);
println!("linspace(0, 1, 11) = {:?}", xs);

// seq: generate a sequence with a given step
let ys = f64::seq(0.0, 2.0, 0.5);
println!("seq(0, 2, 0.5)     = {:?}", ys);

// cumsum: cumulative sum
let data = vec![1.0, 2.0, 3.0, 4.0, 5.0];
let cs = f64::cumsum(&data);
println!("cumsum([1..5])     = {:?}", cs);

// rep: repeat a value
let repeated = f64::rep(3.14, 5);
println!("rep(3.14, 5)       = {:?}", repeated);

## Summary

| Tool | Method | Accuracy |
|------|--------|----------|
| Integration | Tanh-Sinh quadrature | Machine precision |
| Optimization | Gradient Descent + AD | Exact gradients |
| Curves | Linear/Exponential interpolation | Piecewise |
| Sequences | linspace, seq, cumsum, rep | Exact |